# Dataset-agnostic pixel-space manifold geometry — MNIST / CIFAR / known-dimension images

**Research prototype, not a validated universal estimator.** The script learns a noise-conditioned **scalar EBM** by denoising score matching, then estimates a low-dimensional *tangent* space by randomized Jacobian-vector products of the Tweedie denoiser. It **does not materialize the full pixel Hessian**. It automatically searches a noise-scale grid and can **ABSTAIN**.

**Run order:** (1) Run the analytic high-dimensional check, (2) set `DATASET = 'synthetic_blobs'` and evaluate projection error against an exact 2D tangent, (3) change `DATASET = 'mnist1'`, retrain and inspect unseen samples, (4) only then try `cifar10`. The geometry estimator uses the same code on all datasets. No MNIST/CIFAR ground-truth manifold dimension is assumed.

**Key caution:** a score model may have a low DSM loss yet an inaccurate *derivative*; noisy images are not proven to form a single smooth manifold. Unknown or ambiguous spectral rank should be reported as ABSTAIN, not forced to a number.

In [ ]:
# Cell 1: setup (PyTorch, numpy, scipy, torchvision are included in Colab)
import os, math, copy, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import datasets, transforms

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 123
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE.type == 'cuda': torch.cuda.manual_seed_all(SEED)
print('Torch:',torch.__version__,'Device:',DEVICE)

# Change only DATASET to reuse the method. 'synthetic_blobs' has exact d=2 for independent audit.
DATASET = 'synthetic_blobs'   # 'synthetic_blobs', 'mnist1', 'mnist', 'cifar10'
CIFAR_CLASS = None            # e.g. 3, or None = unconditional mixture
TRAIN_STEPS = 2500           # increase for MNIST/CIFAR (e.g., 5000-10000)
BATCH_SIZE = 48
TRAIN_LIMIT = 10000
SIGMA_GRID_SIZE = 7
N_CAL_ANCHORS = 4
N_TEST_ANCHORS = 6
PROBES_PER_SPLIT = 96        # two independent 96-vector sketches; detectable d < 92
PROBES_CHUNK = 16
RISK_SAMPLES = 512
OUT = Path('/content/highd_manifold_results') if Path('/content').exists() else Path('/mnt/data/highd_manifold_results')
OUT.mkdir(parents=True, exist_ok=True)

## Step A — Known-dimension image manifold and real datasets

`synthetic_blobs` renders 28×28 images with only two continuous parameters, the center coordinates. This gives a nonlinear two-dimensional manifold in **784-dimensional image space**, with an analytic tangent reference that is **not used by the selector**. MNIST and CIFAR are loaded through the same dataset interface; their true dimensions are unknown.

In [ ]:
# Cell 2: data loading and analytic tangent for the synthetic control

def render_blob(positions, H=28, W=28, width=0.25):
    # positions: (N,2) continuously varying (cx,cy) in [-0.6, 0.6]
    yy,xx = torch.meshgrid(torch.linspace(-1,1,H,device=positions.device,dtype=positions.dtype),
                           torch.linspace(-1,1,W,device=positions.device,dtype=positions.dtype), indexing='ij')
    dist = (xx[None]-positions[:,0,None,None])**2 + (yy[None]-positions[:,1,None,None])**2
    return torch.exp(-dist/(2*width**2))[:,None,:,:]

def load_image_data(name, max_train=10000):
    transform=transforms.ToTensor()
    if name=='synthetic_blobs':
        gen = torch.Generator().manual_seed(SEED)
        n = max(1024,max_train)
        a = 1.2*torch.rand(n+512,2,generator=gen)-0.6
        x = render_blob(a)
        return x[:n],x[n:n+256],x[n+256:n+512], a[n+256:n+512]
    if name.startswith('mnist'):
        ds_train=datasets.MNIST('./datasets',train=True,download=True,transform=transform)
        ds_test=datasets.MNIST('./datasets',train=False,download=True,transform=transform)
        tr=ds_train.data.float().unsqueeze(1)/255
        te=ds_test.data.float().unsqueeze(1)/255
        if name=='mnist1':
            tr=tr[ds_train.targets==1]
            te=te[ds_test.targets==1]
        return tr[:max_train].contiguous(),te[:512].contiguous(),te[512:768].contiguous(),None
    if name=='cifar10':
        ds_train=datasets.CIFAR10('./datasets',train=True,download=True)
        ds_test=datasets.CIFAR10('./datasets',train=False,download=True)
        tr=torch.as_tensor(np.asarray(ds_train.data)).permute(0,3,1,2).float()/255
        te=torch.as_tensor(np.asarray(ds_test.data)).permute(0,3,1,2).float()/255
        if CIFAR_CLASS is not None:
            tr=tr[np.asarray(ds_train.targets)==CIFAR_CLASS]
            te=te[np.asarray(ds_test.targets)==CIFAR_CLASS]
        return tr[:max_train].contiguous(),te[:512].contiguous(),te[512:768].contiguous(),None
    raise ValueError('Unknown DATASET')

X_train, X_val, X_test, PARAMS_TEST = load_image_data(DATASET,TRAIN_LIMIT)
C,H,W = X_train.shape[1:]; D=C*H*W
print('Dataset:',DATASET,'Train/val/test',len(X_train),len(X_val),len(X_test),'Ambient D:',D)

# Generate a data-scaled noise grid without using any known d or manifold equation.
def scale_grid(X, n=7):
    gen=torch.Generator().manual_seed(994)
    v=X[torch.randperm(len(X),generator=gen)[:min(512,len(X))]].reshape(-1,D).float()
    dist=torch.cdist(v,v) / math.sqrt(D)
    dist.fill_diagonal_(float('inf'))
    nn = dist.min(1).values.median().item()
    floor = max(0.008,0.18*nn)
    cap = min(0.45,max(4*floor,2.8*nn))
    if cap<=floor: cap=4*floor
    result=np.geomspace(floor,cap,n)
    return result,nn

SIGMAS,NN_SCALE=scale_grid(X_train,SIGMA_GRID_SIZE)
print('Per-pixel nearest-neighbor scale:',round(NN_SCALE,4))
print('Automatic candidate sigmas:',np.round(SIGMAS,4))
# These scale multipliers are fixed algorithm design choices, not a proof of optimality.

## Step B — Noise-conditioned scalar EBM trained with DSM

The network outputs **one energy scalar per image**, not a score vector. The score is computed as \(s_\theta=-\nabla_x E_\theta\). It is trained with the weighted per-coordinate DSM loss \(\|\sigma s_\theta(x+\sigma\epsilon,\sigma)+\epsilon\|^2/D\). Different noise levels are sampled log-uniformly. This step needs a GPU for practical runtimes.

In [ ]:
# Cell 3: differentiable scalar-energy model
class EnergyCNN(nn.Module):
    def __init__(self,channels=1,height=28,width=28,width_factor=32):
        super().__init__()
        a=width_factor
        self.features=nn.Sequential(
          nn.Conv2d(channels+1,a,3,padding=1),nn.SiLU(),
          nn.Conv2d(a,a,3,stride=2,padding=1),nn.SiLU(),
          nn.Conv2d(a,2*a,3,stride=2,padding=1),nn.SiLU(),
          nn.Conv2d(2*a,2*a,3,padding=1),nn.SiLU(),
          nn.AdaptiveAvgPool2d(1))
        self.out=nn.Sequential(nn.Flatten(),nn.Linear(2*a,64),nn.SiLU(),nn.Linear(64,1))
        self.D=channels*height*width
    def forward(self,x,sigma):
        if not torch.is_tensor(sigma): sigma=torch.tensor(sigma,dtype=x.dtype,device=x.device)
        if sigma.ndim==0: sigma=sigma.expand(x.shape[0])
        if sigma.ndim>1: sigma=sigma.reshape(-1)
        level=(torch.log(sigma.clamp_min(1e-8))/4).view(-1,1,1,1).expand(-1,1,x.shape[2],x.shape[3])
        # Scale counters global spatial pooling, keeping energy gradients observable.
        return self.out(self.features(torch.cat([x,level],dim=1))).flatten()*self.D

class LearnedScore:
    def __init__(self,model): self.model=model
    def score(self,x,sigma):
        with torch.enable_grad():
            q=x.detach().requires_grad_(True)
            E=self.model(q,sigma)
            s=-torch.autograd.grad(E.sum(),q,create_graph=False)[0]
        return s.detach()

def train_ebm(model,X,steps,batch_size=48,lr=2e-4):
    model.train()
    X=X.cpu()
    optim=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-6)
    losses=[]
    lo,hi=math.log(SIGMAS[0]),math.log(SIGMAS[-1])
    rng=torch.Generator().manual_seed(SEED+17)
    for step in range(steps):
        ids=torch.randint(len(X),(batch_size,),generator=rng)
        clean=X[ids].to(DEVICE)
        sigma=torch.exp(torch.empty(batch_size,device=DEVICE).uniform_(lo,hi))
        eps=torch.randn_like(clean)
        noisy=(clean+sigma[:,None,None,None]*eps).detach().requires_grad_(True)
        E=model(noisy,sigma)
        score=-torch.autograd.grad(E.sum(),noisy,create_graph=True)[0]
        loss=((sigma[:,None,None,None]*score+eps)**2).flatten(1).mean()
        optim.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(),10.0)
        optim.step()
        losses.append(float(loss.detach()))
        if (step+1)%max(1,steps//8)==0:
            print(f'train step {step+1:>5}/{steps}, weighted DSM/coordinate={np.mean(losses[-100:]):.4f}')
    model.eval()
    return losses

MODEL_FILE=OUT / f'{DATASET}_class{CIFAR_CLASS}_N{len(X_train)}_steps{TRAIN_STEPS}_energy_model.pt'
model=EnergyCNN(C,H,W).to(DEVICE)
if MODEL_FILE.exists():
    checkpoint=torch.load(MODEL_FILE,map_location=DEVICE,weights_only=False)
    if checkpoint.get('sigmas') is not None and np.allclose(checkpoint['sigmas'],SIGMAS):
        model.load_state_dict(checkpoint['weights']); print('Loaded matching checkpoint',MODEL_FILE)
    else:
        print('Found different grid; training a new model.')
        losses=train_ebm(model,X_train,TRAIN_STEPS,BATCH_SIZE)
        torch.save({'weights':model.state_dict(),'sigmas':SIGMAS,'dataset':DATASET},MODEL_FILE)
else:
    losses=train_ebm(model,X_train,TRAIN_STEPS,BATCH_SIZE)
    torch.save({'weights':model.state_dict(),'sigmas':SIGMAS,'dataset':DATASET},MODEL_FILE)
model.eval()
SCORE=LearnedScore(model)
print('Weights:',MODEL_FILE)

## Step C — Matrix-free tangent-Jacobian sketch

Tweedie's denoiser \(T_\sigma(x)=x+\sigma^2s_\theta(x,\sigma)\). Its Jacobian is \(J_T=I-\sigma^2\nabla_x^2 E_\theta\). Instead of building a \(D\times D\) matrix, approximate the product \(J_Tv\) with two score evaluations and collect the results in a \(D\times m\) sketch. The small matrix \(m\times m\) is decomposed to estimate **tangent** directions. Important: if true d is at or above m, the method cannot resolve it and must ABSTAIN.

In [ ]:
# Cell 4: efficient finite-difference Jacobian-vector products and randomized spectrum
@torch.no_grad()
def get_device(x): return x.device

def tangent_action_columns(score_model,x,sigma,V,chunk=16):
    # x: (C,H,W), V: (D,m) iid N(0,1) columns; output D x m.
    x=x.to(DEVICE).float(); V=V.to(DEVICE).float()
    m=V.shape[1]
    h=min(0.01,0.15*float(sigma))
    outs=[]
    for i in range(0,m,chunk):
        v=V[:,i:i+chunk].T.contiguous().reshape(-1,C,H,W)
        x0=x.unsqueeze(0).expand_as(v)
        sig=torch.full((len(v),),float(sigma),device=DEVICE)
        plus=score_model.score(x0+h*v,sig).reshape(len(v),-1)
        minus=score_model.score(x0-h*v,sig).reshape(len(v),-1)
        jv=v.reshape(len(v),-1)+(float(sigma)**2)*(plus-minus)/(2*h)
        outs.append(jv.T.detach().cpu())
    return torch.cat(outs,dim=1)

def leading_spectrum(Y):
    # Avoid a D x D covariance. Work with m x m Gram matrix.
    m=Y.shape[1]
    gram=(Y.T@Y)/m
    eig,R=torch.linalg.eigh(gram)
    eig=eig.clamp_min(0).flip(0); R=R.flip(1)
    sv=torch.sqrt(eig)
    return sv,R

def relative_rank(Y, max_fraction=0.90):
    sv,R=leading_spectrum(Y)
    m=len(sv)
    # All constants are fixed before viewing unknown image dimensions.
    kmax=max(1,min(m-4,int(m*max_fraction)))
    ratios=(sv[:kmax]/sv[1:kmax+1].clamp_min(1e-7))
    admissible=(ratios>=1.80) & (sv[1:kmax+1]/sv[0].clamp_min(1e-7) <=0.40)
    candidates=torch.where(admissible)[0]
    if len(candidates)==0 or sv[0]<0.025:
        return None,sv,None,0.0
    # The largest relative gap subject to a small tail and capacity margin.
    idx=candidates[torch.argmax(ratios[candidates])].item()
    k=int(idx+1)
    if k >= kmax-2: return None,sv,None,0.0
    W=R[:,:k]
    U=Y@W
    U=torch.linalg.qr(U,mode='reduced').Q
    return k,sv,U,float(ratios[idx])

def subspace_discrepancy(U,V):
    if U is None or V is None or U.shape[1]!=V.shape[1]: return float('inf')
    k=U.shape[1]
    c=(U.T@V).square().sum().clamp(max=float(k)).item()
    return float(math.sqrt(max(0.0,1-c/k))) # sin-theta Frobenius normalized

# Score/denoiser interface test: a known Gaussian d-dimensional linear manifold in R^784.
class LinearGaussianOracle:
    def __init__(self,U,tau=1.0):
        self.U=U.to(DEVICE); self.tau=float(tau)
    def score(self,x,sigma):
        shape=x.shape
        q=x.reshape(len(x),-1)
        s=sigma.reshape(-1,1)
        tangent=(q@self.U)@self.U.T
        return (-q/s**2 + (self.tau**2/(s**2*(self.tau**2+s**2)))*tangent).reshape(shape)

def oracle_self_test():
    d0=8
    A=torch.randn(D,d0,generator=torch.Generator().manual_seed(22))
    U=torch.linalg.qr(A,mode='reduced').Q
    oracle=LinearGaussianOracle(U)
    x=torch.zeros(C,H,W)
    V=torch.randn(D,48,generator=torch.Generator().manual_seed(5))
    Y=tangent_action_columns(oracle,x,0.08,V,chunk=16)
    k,sv,B,gap=relative_rank(Y)
    err=subspace_discrepancy(B,U) if k==d0 else float('inf')
    print('Known-D analytical sanity check: true d=',d0,'recovered=',k,'tangent error=',round(err,5),'gap=',round(gap,2))
    assert k==d0 and err<0.03, 'Analytic estimator self test failed'

oracle_self_test()

## Step D — Automatic noise-window decision and held-out DSM risk

For each *unseen query image*, two independent random probe sketches estimate tangent rank and basis. The selector requires a spectral gap, matched ranks and projectors between probe splits, agreement with an independently estimated held-out DSM-risk dimension, and consistency across **three neighboring noise scales**. It uses **no true d**. It is conservative and often returns ABSTAIN when the score Hessian is inaccurate. Rank `d < m` is essential; hitting the probe budget is not accepted as proof of dimension.

In [ ]:
# Cell 5: held-out DSM risk and local multi-scale decision

def risk_dim_per_sigma(score_model,images,sigma,batch=32,seed=88):
    # Total (not per-coordinate) weighted DSM risk. Ideal small-noise limit is d.
    gen=torch.Generator().manual_seed(seed)
    vals=[]
    for i in range(0,len(images),batch):
        x=images[i:i+batch].to(DEVICE)
        e=torch.randn(x.shape,generator=gen).to(DEVICE)
        s=torch.full((len(x),),float(sigma),device=DEVICE)
        y=x+sigma*e
        score=score_model.score(y,s)
        v=((sigma*score+e)**2).flatten(1).sum(1)
        vals.extend(v.detach().cpu().numpy().tolist())
    vals=np.asarray(vals)
    return float(np.mean(vals)),float(np.std(vals,ddof=1)/np.sqrt(max(1,len(vals))))

# This is an independent diagnostic, not an oracle: imperfect scores can inflate risk.
RISK_LOOKUP={}
for sigma in SIGMAS:
    RISK_LOOKUP[float(sigma)]=risk_dim_per_sigma(SCORE,X_val[:RISK_SAMPLES],float(sigma))
print('Heldout weighted DSM-risk dimension proxy:')
display(pd.DataFrame([dict(sigma=k,risk_d=v[0],risk_se=v[1]) for k,v in RISK_LOOKUP.items()]).round(3))

def estimate_one(score_model,x,sigma_grid,seed=123,probes=96,risk_lookup=None):
    rng=torch.Generator().manual_seed(seed)
    V1=torch.randn(D,probes,generator=rng)
    V2=torch.randn(D,probes,generator=rng)
    records=[]
    for sig in sigma_grid:
        Y1=tangent_action_columns(score_model,x,float(sig),V1,PROBES_CHUNK)
        Y2=tangent_action_columns(score_model,x,float(sig),V2,PROBES_CHUNK)
        d1,sv1,U1,g1=relative_rank(Y1)
        d2,sv2,U2,g2=relative_rank(Y2)
        disc=subspace_discrepancy(U1,U2)
        agree=(d1 is not None and d1==d2 and disc<=0.22)
        # If d saturates probe count, we do not pretend to know the true rank.
        risk,se=(float('nan'),float('nan')) if risk_lookup is None else risk_lookup[float(sig)]
        risk_ok=(risk_lookup is None or (agree and abs(risk-d1)<=max(6.0,0.35*d1,3.0*se)))
        records.append(dict(sigma=float(sig),dimension=d1,split_dimension=d2,split_proj_error=disc,
                            gap1=g1,gap2=g2,lead_singular=float(sv1[0]),risk_d=risk,
                            risk_se=se,good=bool(agree and risk_ok),basis=U1))
    # Search contiguous >=3-scales good blocks with the same estimated tangent rank.
    blocks=[]; start=None
    for i,row in enumerate(records+[dict(good=False,dimension=None)]):
        if row['good'] and (start is None or row['dimension']==records[i-1]['dimension']):
            if start is None: start=i
        else:
            if start is not None and i-start>=3:
                sub=records[start:i]
                adjacent=[subspace_discrepancy(sub[j]['basis'],sub[j+1]['basis']) for j in range(len(sub)-1)]
                if max(adjacent,default=float('inf'))<=0.22:
                    blocks.append((start,i,sub,adjacent))
            start=i if row['good'] else None
    if not blocks:
        decision={'status':'ABSTAIN','dimension':None,'reason':'no 3-scale stable and risk-consistent tangent eigenspace'}
    else:
        # Prefer longest window; among ties choose smaller mean split discrepancy.
        a,b,sub,_=max(blocks,key=lambda z:(z[1]-z[0],-np.mean([r['split_proj_error'] for r in z[2]])))
        decision={'status':'PASS','dimension':int(sub[0]['dimension']),
                  'window':(float(sub[0]['sigma']),float(sub[-1]['sigma'])),
                  'n_scales':len(sub),'basis':sub[len(sub)//2]['basis']}
    table=pd.DataFrame([{k:v for k,v in r.items() if k!='basis'} for r in records])
    return decision,table

## Step E — Unseen-image evaluation and optional ground-truth tangent audit

This is where we test whether the estimator really works. For `synthetic_blobs`, we independently obtain the exact 2D tangent by differentiating the known renderer, and compute subspace error **after** the decision. For MNIST/CIFAR, no oracle exists, so we report accepted/abstained estimates and spectral diagnostics, not fabricated ground-truth accuracy. Save CSV diagnostics and visualize queries.

In [ ]:
# Cell 6: final evaluation on unseen images

def true_blob_basis(p):
    # Only used as an audit, never for selecting scale or rank.
    p=p.detach().clone().to(DEVICE).requires_grad_(True)
    def f(v): return render_blob(v[None]).reshape(-1)
    J=torch.autograd.functional.jacobian(f,p).detach().cpu()
    return torch.linalg.qr(J,mode='reduced').Q

def evaluate_unseen(count=N_TEST_ANCHORS):
    summaries=[]; tables=[]
    for i in range(min(count,len(X_test))):
        x=X_test[i]
        result,tab=estimate_one(SCORE,x,SIGMAS,seed=1000+i,probes=PROBES_PER_SPLIT,risk_lookup=RISK_LOOKUP)
        d_true=2 if DATASET=='synthetic_blobs' else None
        error=None
        if DATASET=='synthetic_blobs' and result['status']=='PASS':
            exact=true_blob_basis(PARAMS_TEST[i])
            error=subspace_discrepancy(result['basis'],exact) if result['dimension']==2 else float('inf')
        summary=dict(anchor=i,status=result['status'],dimension=result.get('dimension'),
                     window_lo=result.get('window',(None,None))[0],window_hi=result.get('window',(None,None))[1],
                     oracle_true_d=d_true,oracle_tangent_error=error)
        summaries.append(summary)
        tab.insert(0,'anchor',i); tables.append(tab)
        print('unseen anchor',i,summary)
    summary=pd.DataFrame(summaries)
    detail=pd.concat(tables,ignore_index=True)
    summary.to_csv(OUT/f'{DATASET}_summary.csv',index=False)
    detail.to_csv(OUT/f'{DATASET}_scales.csv',index=False)
    return summary,detail

summary,detail=evaluate_unseen()
display(summary)

fig,axes=plt.subplots(1,min(6,N_TEST_ANCHORS,len(X_test)),figsize=(2.4*min(6,N_TEST_ANCHORS,len(X_test)),3))
axes=np.atleast_1d(axes)
for idx,ax in enumerate(axes):
    q=X_test[idx]
    if C==1: ax.imshow(q[0],cmap='gray',vmin=0,vmax=1)
    else: ax.imshow(q.permute(1,2,0))
    r=summary.iloc[idx]
    ax.set_title(f"#{idx}: {r['status']}\nd={r['dimension']}",fontsize=10)
    ax.axis('off')
plt.tight_layout();plt.savefig(OUT/f'{DATASET}_queries.png',dpi=160);plt.show()
print('Outputs saved to:',OUT)

## Interpretation and next decision

- On `synthetic_blobs`, **the expected true dimension is 2**. If the estimator does not pass and recover approximately the analytic tangent space, diagnose the score model before moving to MNIST.
- On `mnist1` / `cifar10`, a reported local dimension is only a **model-dependent estimate**, not verified ground truth. `ABSTAIN` indicates the learned score's tangent Jacobian lacks a stable, reliably identifiable rank. Increase training quality and compare against a published diffusion-score baseline before changing thresholds.
- **Probe-capacity constraint:** each independent sketch contains `PROBES_PER_SPLIT` columns. If the underlying dimension is at least that many, the full dimension is unidentifiable with these probes. Increase the automatic computational budget or add a different estimator; do not claim the truncated rank is the dimension.
- A further validity check is multiple independent *trained models* and unseen data splits, not just repeat perturbation seeds.
- **No class-label information or true d** is used in the geometric selection, but `mnist1` and a class-filtered CIFAR run are class-conditional training tasks by design.

**Mathematical limitation:** for every finite \(\sigma\), the Tweedie denoiser derivative need not be an exact tangent projector. Its spectrum may have no clean gap, and no algorithm can guarantee identification of arbitrary unknown manifolds from finite images. This notebook is designed to expose that failure rather than hide it.